# Redes Neuronales: Generalización de la Regresión Logística

En esta clase veremos cómo una red neuronal puede entenderse como una **generalización** de la regresión logística. 

La regresión logística es un clasificador lineal que aprende una frontera de decisión usando una sola neurona con activación sigmoide.

Al apilar varias neuronas (capas ocultas), obtenemos un **perceptrón multicapa** (MLP) capaz de aprender relaciones no lineales.

Implementaremos ejemplos simples en **TensorFlow / Keras** y en **PyTorch**.

## Regresión Logística como una neurona

La regresión logística tiene la forma:

$$\hat{y} = \sigma(\mathbf{w}^T \mathbf{x} + b)$$

donde $\sigma(z) = \frac{1}{1+e^{-z}}$. Esto es exactamente una **neurona** con activación sigmoide.

Una red neuronal extiende esta idea usando múltiples neuronas organizadas en capas.

Vamos a implementar:
1. Un modelo de regresión logística (una sola neurona) → equivalente a `LogisticRegression` de sklearn.
2. Un MLP con una capa oculta.

Esto mostrará cómo la red neuronal puede capturar patrones más complejos.

In [ ]:
# Importar librerías comunes
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generar dataset sintético
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0, 
                           n_informative=2, n_clusters_per_class=1, 
                           random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Visualizar datos
plt.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolor='k')
plt.title("Datos sintéticos (2 clases)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

## 1. Regresión Logística con TensorFlow / Keras

En Keras, una regresión logística es simplemente una capa `Dense(1, activation='sigmoid')`.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Modelo de regresión logística (una neurona)
model_logistic = Sequential([
    Dense(1, activation='sigmoid', input_shape=(2,))
])

# Compilar
model_logistic.compile(optimizer=Adam(learning_rate=0.01),
                       loss='binary_crossentropy',
                       metrics=['accuracy'])

# Entrenar
history_logistic = model_logistic.fit(X_train, y_train, epochs=50, verbose=0, validation_split=0.2)

# Evaluar
y_pred_logistic = (model_logistic.predict(X_test) > 0.5).astype(int)
acc_logistic = accuracy_score(y_test, y_pred_logistic)
print(f"Precisión regresión logística (Keras): {acc_logistic:.4f}")

## 2. Red Neuronal con una capa oculta (TensorFlow/Keras)

Añadimos una capa oculta con 10 neuronas y activación ReLU. Esto es un **MLP** que puede aprender separaciones no lineales.

In [ ]:
model_mlp = Sequential([
    Dense(10, activation='relu', input_shape=(2,)),
    Dense(1, activation='sigmoid')
])

model_mlp.compile(optimizer=Adam(learning_rate=0.01),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

history_mlp = model_mlp.fit(X_train, y_train, epochs=50, verbose=0, validation_split=0.2)

y_pred_mlp = (model_mlp.predict(X_test) > 0.5).astype(int)
acc_mlp = accuracy_score(y_test, y_pred_mlp)
print(f"Precisión MLP (Keras): {acc_mlp:.4f}")

## 3. Regresión Logística con PyTorch

PyTorch requiere definir explícitamente el modelo, la función de pérdida y el optimizador, así como el bucle de entrenamiento.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Convertir datos a tensores
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

# Dataset y DataLoader
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Definir modelo (regresión logística)
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.linear(x))

model_logistic_pt = LogisticRegression(2)
criterion = nn.BCELoss()
optimizer = optim.Adam(model_logistic_pt.parameters(), lr=0.01)

# Entrenamiento
epochs = 50
for epoch in range(epochs):
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_logistic_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Evaluación
with torch.no_grad():
    y_pred_logistic_pt = model_logistic_pt(X_test_t).numpy() > 0.5
acc_logistic_pt = accuracy_score(y_test, y_pred_logistic_pt.astype(int))
print(f"Precisión regresión logística (PyTorch): {acc_logistic_pt:.4f}")

## 4. Red Neuronal con una capa oculta (PyTorch)

Añadimos una capa lineal intermedia con activación ReLU.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=10):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.relu(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

model_mlp_pt = MLP(2, hidden_dim=10)
criterion = nn.BCELoss()
optimizer = optim.Adam(model_mlp_pt.parameters(), lr=0.01)

# Entrenamiento
for epoch in range(epochs):
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_mlp_pt(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Evaluación
with torch.no_grad():
    y_pred_mlp_pt = model_mlp_pt(X_test_t).numpy() > 0.5
acc_mlp_pt = accuracy_score(y_test, y_pred_mlp_pt.astype(int))
print(f"Precisión MLP (PyTorch): {acc_mlp_pt:.4f}")

## Comparación de resultados

| Modelo | Precisión |
|--------|-----------|
| Regresión Logística (Keras) | {acc_logistic:.4f} |
| MLP (Keras) | {acc_mlp:.4f} |
| Regresión Logística (PyTorch) | {acc_logistic_pt:.4f} |
| MLP (PyTorch) | {acc_mlp_pt:.4f} |

Observamos que el MLP (con una capa oculta) puede lograr mayor precisión que la regresión logística en este conjunto de datos, ya que puede aprender fronteras de decisión no lineales.

**Conclusión**: Las redes neuronales generalizan la regresión logística al apilar múltiples neuronas y capas, permitiendo modelar relaciones complejas.

## Visualización de la frontera de decisión

Graficamos la frontera aprendida por el MLP de PyTorch (se puede hacer similar con Keras).

In [ ]:
def plot_decision_boundary(model, X, y, title=""):
    # Crear malla
    x_min, x_max = X[:,0].min() - 0.5, X[:,0].max() + 0.5
    y_min, y_max = X[:,1].min() - 0.5, X[:,1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))
    grid = np.c_[xx.ravel(), yy.ravel()]
    with torch.no_grad():
        Z = model(torch.tensor(grid, dtype=torch.float32)).numpy()
    Z = Z.reshape(xx.shape)
    plt.contourf(xx, yy, Z, levels=[0,0.5,1], cmap='coolwarm', alpha=0.3)
    plt.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolor='k')
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

plot_decision_boundary(model_mlp_pt, X, y, title="Frontera MLP (PyTorch)")

## Ejercicio para el estudiante

1. Modifique la arquitectura del MLP (número de neuronas, número de capas) y observe cómo cambia la precisión.
2. Pruebe diferentes funciones de activación (tanh, leaky_relu).
3. Implemente **dropout** para regularización y note su efecto.
4. Compare el rendimiento con un dataset más complejo (por ejemplo, `make_moons`).